In [2]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv("../data/telco-customer-churn.csv")
df = df.drop(columns=["customerID"])

In [4]:
df["TotalCharges"] = df["TotalCharges"].replace(" ",0).astype(float)

In [ ]:
df["Churn"] = df["Churn"].map({"Yes": 1.0, "No": 0.0})

In [6]:
df.to_csv("../data/telco-customer-churn-cleaned.csv")

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# 1. 데이터 로드 및 전처리
df = pd.read_csv("../data/telco-customer-churn.csv")

# 고유 식별자 컬럼 제거
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

# 공백 데이터 및 타입 에러 처리 (TotalCharges)
df["TotalCharges"] = df["TotalCharges"].replace(" ", "0")
df["TotalCharges"] = df["TotalCharges"].astype(float)

# 타겟 변수 정수형 이진화 (0과 1)
y = df['Churn'].map({"Yes": 1, "No": 0})
X = df.drop(columns=['Churn'])

# 데이터셋 내의 모든 범주형(object) 변수 자동 탐색
# (InternetService, Contract 외에 Gender, Partner, PaymentMethod 등이 모두 포함됩니다)
cat_features = X.select_dtypes(include=['object']).columns.tolist()


# 2. 모델별 데이터 전처리 (원핫 인코딩 vs 카테고리 타입 변환)

# [XGBoost 전처리] 
# 1) 모든 범주형 변수를 원핫 인코딩
X_encoded = pd.get_dummies(X, columns=cat_features, drop_first=True)
# 2) XGBoost 최신 버전 호환성을 위해 bool 타입을 int/float 형태로 일괄 변환
X_encoded = X_encoded.astype(float)

X_train_xgb, X_test_xgb, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)


# [CatBoost 전처리]
# 문자열 데이터 타입을 CatBoost가 인식할 수 있도록 'category' 또는 'str'로 명시적 변환
X_cat = X.copy()
for col in cat_features:
    X_cat[col] = X_cat[col].astype(str)

X_train_cat, X_test_cat, _, _ = train_test_split(
    X_cat, y, test_size=0.2, random_state=42, stratify=y
)


# 3. 불균형 데이터셋 가중치 계산 (y가 정수형이어야 bincount 정상 작동)
neg_count, pos_count = np.bincount(y_train)
imbalance_ratio = neg_count / pos_count


# 4. XGBoost 모델 학습 및 평가
print("=== XGBoost Classification ===")
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    scale_pos_weight=imbalance_ratio, 
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train_xgb, y_train)

y_pred_xgb = xgb_model.predict(X_test_xgb)
print(classification_report(y_test, y_pred_xgb))


# 5. CatBoost 모델 학습 및 평가
print("\n=== CatBoost Classification ===")
# 전처리된 X_cat 데이터프레임 기준으로 범주형 변수 인덱스 재추출
cat_indices = [X_cat.columns.get_loc(col) for col in cat_features]

cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=5,
    scale_pos_weight=imbalance_ratio, 
    cat_features=cat_indices,          
    random_seed=42,
    verbose=0                          
)
cat_model.fit(X_train_cat, y_train)

y_pred_cat = cat_model.predict(X_test_cat)
print(classification_report(y_test, y_pred_cat))


=== XGBoost Classification ===


              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.52      0.80      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409


=== CatBoost Classification ===
              precision    recall  f1-score   support

           0       0.91      0.74      0.81      1035
           1       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409

